# Notebook 1: Data Setup

**Purpose:** Download and preprocess EyePACS and APTOS datasets for the stochastic orthogonality experiment.

**Requirements:**
- Google Colab with GPU runtime
- Kaggle API token (`kaggle.json`)
- Accepted competition rules for both datasets on Kaggle

**Time:** ~2 hours (dominated by EyePACS download)

**Output:** Preprocessed images and train/val/test splits saved to Google Drive

## Step 1: Mount Drive and Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR = '/content/drive/MyDrive/stochastic_orthogonality'

# Create directory structure
for subdir in ['data/eyepacs/raw', 'data/eyepacs_full/processed',
               'data/aptos/raw', 'data/aptos/processed',
               'models_eyepacs_full', 'results_eyepacs_full']:
    os.makedirs(f'{BASE_DIR}/{subdir}', exist_ok=True)
    
print('Directory structure created.')

## Step 2: Upload Kaggle API Token

Run this cell and upload your `kaggle.json` file when prompted.

In [ ]:
from google.colab import files
import json

print('Upload your kaggle.json file:')
uploaded = files.upload()

# Set up Kaggle credentials
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    f.write(list(uploaded.values())[0].decode())
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials configured.')

## Step 3: Install Dependencies

In [ ]:
!pip install -q kaggle pillow pandas scikit-learn tqdm

## Step 4: Download APTOS 2019 (~1 GB, fast)

In [ ]:
os.chdir(f'{BASE_DIR}/data/aptos/raw')
!kaggle competitions download -c aptos2019-blindness-detection
print('\nAPTOS download complete. Extracting...')
!unzip -q -o aptos2019-blindness-detection.zip
print('Done.')

## Step 5: Download EyePACS (~80 GB, 1-2 hours)

This is a large download. Files save to Drive, so progress survives Colab disconnects.

In [ ]:
os.chdir(f'{BASE_DIR}/data/eyepacs/raw')
!kaggle competitions download -c diabetic-retinopathy-detection
print('\nEyePACS download complete.')

## Step 6: Extract and Preprocess Images

Resize all images to 512×512 and save as PNG.

In [ ]:
from PIL import Image
from tqdm import tqdm
import pandas as pd
import glob

TARGET_SIZE = (512, 512)

def preprocess_images(input_dir, output_dir, target_size=TARGET_SIZE):
    os.makedirs(output_dir, exist_ok=True)
    existing = set(os.path.splitext(f)[0] for f in os.listdir(output_dir) if f.endswith('.png'))
    
    image_files = glob.glob(os.path.join(input_dir, '*.jpeg')) + glob.glob(os.path.join(input_dir, '*.png'))
    print(f'Found {len(image_files)} images, {len(existing)} already processed')
    
    for img_path in tqdm(image_files):
        img_id = os.path.splitext(os.path.basename(img_path))[0]
        if img_id in existing:
            continue
        try:
            img = Image.open(img_path).convert('RGB')
            img = img.resize(target_size, Image.LANCZOS)
            img.save(os.path.join(output_dir, f'{img_id}.png'))
        except Exception as e:
            print(f'Error processing {img_id}: {e}')

# Preprocess APTOS
print('Preprocessing APTOS...')
preprocess_images(f'{BASE_DIR}/data/aptos/raw/train_images',
                  f'{BASE_DIR}/data/aptos/processed')

# Preprocess EyePACS (adjust input path based on extraction)
print('\nPreprocessing EyePACS...')
eyepacs_raw = f'{BASE_DIR}/data/eyepacs/raw'
for subdir in ['train', 'test']:
    input_path = os.path.join(eyepacs_raw, subdir)
    if os.path.exists(input_path):
        preprocess_images(input_path, f'{BASE_DIR}/data/eyepacs_full/processed')

print('\nPreprocessing complete.')

## Step 7: Create Train/Val/Test Splits

Binarize labels (grades 0-1 = non-referable, grades 2-4 = referable) and create stratified splits.

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

FULL_DIR = f'{BASE_DIR}/data/eyepacs_full'
PROCESSED_DIR = f'{FULL_DIR}/processed'

# Load EyePACS labels
labels_df = pd.read_csv(f'{BASE_DIR}/data/eyepacs/raw/trainLabels.csv')
labels_df.columns = ['image_id', 'level']
labels_df['binary_label'] = (labels_df['level'] >= 2).astype(int)

# Filter to existing processed images
existing = set(os.path.splitext(f)[0] for f in os.listdir(PROCESSED_DIR) if f.endswith('.png'))
labels_df = labels_df[labels_df['image_id'].isin(existing)].reset_index(drop=True)
print(f'Total images with labels: {len(labels_df)}')
print(f'Grade distribution:\n{labels_df["level"].value_counts().sort_index()}')

# Stratified 80/10/10 split
train_df, temp_df = train_test_split(labels_df, test_size=0.2, 
                                      stratify=labels_df['level'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5,
                                    stratify=temp_df['level'], random_state=42)

# Save splits
train_df.to_csv(f'{FULL_DIR}/eyepacs_full_train.csv', index=False)
val_df.to_csv(f'{FULL_DIR}/eyepacs_full_val.csv', index=False)
test_df.to_csv(f'{FULL_DIR}/eyepacs_full_test.csv', index=False)

print(f'\nSplits saved:')
print(f'  Train: {len(train_df)} ({train_df["binary_label"].mean():.1%} referable)')
print(f'  Val:   {len(val_df)} ({val_df["binary_label"].mean():.1%} referable)')
print(f'  Test:  {len(test_df)} ({test_df["binary_label"].mean():.1%} referable)')

## Step 8: Verify

In [ ]:
print('='*60)
print('DATA PREPARATION COMPLETE')
print('='*60)
print()
for split in ['train', 'val', 'test']:
    df = pd.read_csv(f'{FULL_DIR}/eyepacs_full_{split}.csv')
    severe = (df['level'] >= 3).sum()
    print(f'{split}: {len(df)} images, {severe} severe (grade 3+4)')

aptos = pd.read_csv(f'{BASE_DIR}/data/aptos/raw/train.csv')
aptos.columns = ['image_id', 'level']
severe_a = (aptos['level'] >= 3).sum()
print(f'\nAPTOS (external): {len(aptos)} images, {severe_a} severe')
print('\nReady for Notebook 2 (training).')